In [1]:
"""
Compare live flow features (from your Kali -> Lubuntu test) with the CICIDS2017
training data to spot unit or extractor mismatches.

Reading the output:
  orders_off = how many powers of ten the live value is away from the 2017
               median of the TRUE class (about +-3 or +-6 usually means a unit
               problem such as seconds or milliseconds instead of microseconds)
  in_5_95    = whether the live value sits inside the 2017 5th-95th percentile
               range of the true class
"""
import re
import numpy as np
import pandas as pd
import joblib


In [2]:

# ---------------- CONFIG ----------------
LIVE_CSV = r'ftp_patator.csv'     # csv written by your live flow extractor
TRUE_CLASS = 'Brute Force'               # Brute Force | DDoS | DoS | Normal Traffic | Port Scan
ROWS = None                      # None = median of all rows in the file; [0] = first row only; [3, 7] = those rows
PATH_2017 = r'D:\Academic\ML\ModelTraining\Preprocessing\cicids2017_preprocessed.csv'
MODEL = r'D:\Academic\ML\ModelTraining\newOutput\ensemble_model.pkl'
SCALER = r'D:\Academic\ML\ModelTraining\newOutput\scaler.pkl'
ENCODER = r'D:\Academic\ML\ModelTraining\newOutput\label_encoder.pkl'


In [3]:
KEY = ['Flow Duration', 'Flow IAT Mean', 'Flow Packets/s',
       'Fwd Header Length', 'Init_Win_bytes_forward']
# ----------------------------------------

FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min',
    'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max',
    'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
    'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
    'Flow IAT Min', 'Fwd IAT Std', 'Fwd IAT Min', 'Bwd IAT Total',
    'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length',
    'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length',
    'Packet Length Mean', 'Packet Length Std', 'FIN Flag Count',
    'PSH Flag Count', 'ACK Flag Count', 'Init_Win_bytes_forward',
    'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward',
    'Active Mean', 'Active Max', 'Active Min', 'Idle Mean',
]
ABBREV = {
    'Tot Fwd Pkts': 'Total Fwd Packets', 'Tot Bwd Pkts': 'Total Backward Packets',
    'TotLen Fwd Pkts': 'Total Length of Fwd Packets',
    'Fwd Pkt Len Max': 'Fwd Packet Length Max', 'Fwd Pkt Len Min': 'Fwd Packet Length Min',
    'Fwd Pkt Len Mean': 'Fwd Packet Length Mean', 'Fwd Pkt Len Std': 'Fwd Packet Length Std',
    'Bwd Pkt Len Max': 'Bwd Packet Length Max', 'Bwd Pkt Len Mean': 'Bwd Packet Length Mean',
    'Bwd Pkt Len Std': 'Bwd Packet Length Std', 'Flow Byts/s': 'Flow Bytes/s',
    'Flow Pkts/s': 'Flow Packets/s', 'Bwd IAT Tot': 'Bwd IAT Total',
    'Fwd Header Len': 'Fwd Header Length', 'Bwd Header Len': 'Bwd Header Length',
    'Bwd Pkts/s': 'Bwd Packets/s', 'Pkt Len Min': 'Min Packet Length',
    'Pkt Len Max': 'Max Packet Length', 'Pkt Len Mean': 'Packet Length Mean',
    'Pkt Len Std': 'Packet Length Std', 'FIN Flag Cnt': 'FIN Flag Count',
    'PSH Flag Cnt': 'PSH Flag Count', 'ACK Flag Cnt': 'ACK Flag Count',
    'Init Fwd Win Byts': 'Init_Win_bytes_forward',
    'Init Bwd Win Byts': 'Init_Win_bytes_backward',
    'Fwd Act Data Pkts': 'act_data_pkt_fwd', 'Fwd Seg Size Min': 'min_seg_size_forward',
}


In [4]:
def norm(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())


# Match live column names ignoring case, spaces, underscores and slashes,
# so "Flow Duration", "flow_duration" and "flow_byts_s" all resolve.
lookup = {norm(f): f for f in FEATURES}
lookup.update({norm(a): f for a, f in ABBREV.items()})

live = pd.read_csv(LIVE_CSV)
live = live.rename(columns={c: lookup[norm(c)] for c in live.columns if norm(c) in lookup})
missing = [f for f in FEATURES if f not in live.columns]
if missing:
    print(f'WARNING - live file has no column for: {missing}')
    print(f'Live columns found: {list(live.columns)}\n')
feats = [f for f in FEATURES if f in live.columns]

live_num = live[feats].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
sel = live_num.iloc[ROWS] if ROWS is not None else live_num
live_val = sel.median()
print(f'Live rows used: {len(sel)}   true class: {TRUE_CLASS}\n')

d17 = pd.read_csv(PATH_2017, usecols=FEATURES + ['Attack Type'])
cls = d17[d17['Attack Type'] == TRUE_CLASS][feats]
nrm = d17[d17['Attack Type'] == 'Normal Traffic'][feats]
p05, p95 = cls.quantile(0.05), cls.quantile(0.95)

t = pd.DataFrame({
    'live': live_val,
    f'2017 {TRUE_CLASS} median': cls.median(),
    '2017 Normal median': nrm.median(),
})
t['orders_off'] = np.log10(1 + t['live'].abs()) - np.log10(1 + t[f'2017 {TRUE_CLASS} median'].abs())
t['in_5_95'] = (live_val >= p05) & (live_val <= p95)

fmt = lambda v: f'{v:.4g}'
key = [k for k in KEY if k in t.index]
print('KEY FEATURES')
print(t.loc[key].to_string(float_format=fmt))

others = t.drop(index=key)
flagged = others[(others['orders_off'].abs() >= 2) | (~others['in_5_95'])]
print(f'\nOTHER FEATURES outside the 2017 5-95% range or 2+ orders of magnitude off ({len(flagged)} of {len(others)}):')
print(flagged.sort_values('orders_off', key=abs, ascending=False).to_string(float_format=fmt))


Live rows used: 225   true class: Brute Force

KEY FEATURES
                            live  2017 Brute Force median  2017 Normal median  orders_off  in_5_95
Flow Duration          9.265e+06                9.114e+06            4.02e+04    0.007121     True
Flow IAT Mean          3.543e+05                2.507e+05           1.606e+04      0.1503     True
Flow Packets/s             2.929                    4.064                  85     -0.1102     True
Fwd Header Length            328                      296                  64     0.04444     True
Init_Win_bytes_forward 6.424e+04                 2.92e+04                 122      0.3424    False

OTHER FEATURES outside the 2017 5-95% range or 2+ orders of magnitude off (8 of 35):
                             live  2017 Brute Force median  2017 Normal median  orders_off  in_5_95
Init_Win_bytes_backward 6.516e+04                      227                  -1       2.456    False
Fwd Packet Length Min          52                        0  

In [5]:

# Optional: what the saved model says about the same input
try:
    model, scaler, le = joblib.load(MODEL), joblib.load(SCALER), joblib.load(ENCODER)
    if len(feats) == len(FEATURES):
        x = scaler.transform(live_val[FEATURES].fillna(0).values.reshape(1, -1))
        proba = model.predict_proba(x)[0]
        print('\nModel probabilities for this input:')
        for c, p in sorted(zip(le.classes_, proba), key=lambda z: -z[1]):
            print(f'  {c}: {p:.4f}')
    else:
        print('\nSkipped model probabilities - live file is missing some features.')
except Exception as e:
    print(f'\nSkipped model probabilities: {e}')


Model probabilities for this input:
  Normal Traffic: 0.9567
  DoS: 0.0176
  DDoS: 0.0151
  Brute Force: 0.0081
  Port Scan: 0.0025


c:\Users\abhinav\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(


In [6]:
print('\nMin length values in live flows (top 8):')
print(live_num['Min Packet Length'].value_counts().head(8).to_string())

model, scaler, le = joblib.load(MODEL), joblib.load(SCALER), joblib.load(ENCODER)
P = model.predict_proba(scaler.transform(live_num[FEATURES].fillna(0).values))
ni = list(le.classes_).index('Normal Traffic')
print('\nPer-flow predictions:')
print(pd.Series(le.classes_[P.argmax(axis=1)]).value_counts().to_string())
print(pd.Series(P[:, ni]).describe(percentiles=[.05, .25, .5, .75, .95]).round(4).to_string())
for t in (0.5, 0.9, 0.99):
    print(f'flows with P(Normal) < {t}: {(P[:, ni] < t).mean():.3f}')

c:\Users\abhinav\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(



Min length values in live flows (top 8):
Min Packet Length
40     206
52      14
198      2
48       1
310      1
325      1

Per-flow predictions:
Normal Traffic    225
count    225.0000
mean       0.9549
std        0.0082
min        0.9227
5%         0.9423
25%        0.9542
50%        0.9567
75%        0.9567
95%        0.9593
max        0.9875
flows with P(Normal) < 0.5: 0.000
flows with P(Normal) < 0.9: 0.000
flows with P(Normal) < 0.99: 1.000


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler

LENGTH_FEATS = ['Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min',
                'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max',
                'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
                'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std']
MIN_IAT = ['Flow IAT Min', 'Fwd IAT Min', 'Bwd IAT Min']

sample = pd.concat([g.sample(min(len(g), 30000), random_state=42) for _, g in d17.groupby('Attack Type')])
for name, drop in [('A: without length features', LENGTH_FEATS),
                   ('B: without length features and min-IAT features', LENGTH_FEATS + MIN_IAT)]:
    keep = [f for f in FEATURES if f not in drop]
    sc2 = RobustScaler().fit(sample[keep].values)
    rf2 = RandomForestClassifier(n_estimators=100, max_depth=20, class_weight='balanced',
                                 n_jobs=-1, random_state=42).fit(sc2.transform(sample[keep].values),
                                                                  sample['Attack Type'])
    P2 = rf2.predict_proba(sc2.transform(live_num[keep].fillna(0).values))
    ni2 = list(rf2.classes_).index('Normal Traffic')
    print(f'\n{name} ({len(keep)} features) - per-flow predictions on live rows:')
    print(pd.Series(rf2.classes_[P2.argmax(axis=1)]).value_counts().to_string())
    print(f'median P(Normal): {np.median(P2[:, ni2]):.4f}   flows with P(Normal) < 0.9: {(P2[:, ni2] < 0.9).mean():.3f}')


A: without length features (27 features) - per-flow predictions on live rows:
Brute Force       198
Normal Traffic     27
median P(Normal): 0.1800   flows with P(Normal) < 0.9: 0.982

B: without length features and min-IAT features (24 features) - per-flow predictions on live rows:
Brute Force       198
Normal Traffic     27
median P(Normal): 0.2300   flows with P(Normal) < 0.9: 0.987


In [8]:
print((d17['Init_Win_bytes_backward'] == -1).groupby(d17['Attack Type']).mean().round(3))
print((d17['Init_Win_bytes_forward'] == -1).groupby(d17['Attack Type']).mean().round(3))
print('live backward == 0:', (live_num['Init_Win_bytes_backward'] == 0).mean())

Attack Type
Brute Force       0.008
DDoS              0.363
DoS               0.085
Normal Traffic    0.550
Port Scan         0.001
Name: Init_Win_bytes_backward, dtype: float64
Attack Type
Brute Force       0.000
DDoS              0.000
DoS               0.000
Normal Traffic    0.435
Port Scan         0.000
Name: Init_Win_bytes_forward, dtype: float64
live backward == 0: 0.0


In [9]:
z = pd.DataFrame({
    'live zero': (live_num[feats] == 0).mean(),
    '2017 class zero': (cls == 0).mean(),
    'live neg': (live_num[feats] < 0).mean(),
    '2017 class neg': (cls < 0).mean(),
})
z['zero gap'] = (z['live zero'] - z['2017 class zero']).abs()
z['neg gap'] = (z['live neg'] - z['2017 class neg']).abs()
print(z[(z['zero gap'] > 0.3) | (z['neg gap'] > 0.1)].round(3).to_string())

                       live zero  2017 class zero  live neg  2017 class neg  zero gap  neg gap
Fwd Packet Length Min      0.000            0.999       0.0             0.0     0.999      0.0
Min Packet Length          0.000            1.000       0.0             0.0     1.000      0.0
FIN Flag Count             0.067            1.000       0.0             0.0     0.933      0.0
ACK Flag Count             0.022            0.760       0.0             0.0     0.738      0.0


In [10]:
def flag_profile(d):
    pk = d['Total Fwd Packets'] + d['Total Backward Packets']
    has_data = d['act_data_pkt_fwd'] > 0
    return pd.Series({
        'rows': len(d),
        'median packets per flow': pk.median(),
        'PSH==0 among flows with fwd data packets': (d.loc[has_data, 'PSH Flag Count'] == 0).mean(),
        'ACK==0 among flows with 3+ packets': (d.loc[pk >= 3, 'ACK Flag Count'] == 0).mean(),
        'median PSH / packets': (d['PSH Flag Count'] / pk).median(),
        'median ACK / packets': (d['ACK Flag Count'] / pk).median(),
    })

print(pd.DataFrame({f'2017 {TRUE_CLASS}': flag_profile(cls),
                    '2017 Normal': flag_profile(nrm),
                    'live': flag_profile(live_num[feats])}).round(3).to_string())

                                          2017 Brute Force  2017 Normal     live
rows                                              9150.000  2094458.000  225.000
median packets per flow                             24.000        4.000   26.000
PSH==0 among flows with fwd data packets             0.000        0.622    0.022
ACK==0 among flows with 3+ packets                   0.784        0.852    0.009
median PSH / packets                                 0.019        0.000    0.538
median ACK / packets                                 0.000        0.000    0.923
